# Exploratory Data Analysis of the solar eclipse dataset.

## Findings summary

1. The solar dataset contains 11,898 records and 15 columns.
2. No duplicates
3. Missing values associated with partial eclipses, so these rows should not
be removed.

4. The dataset contains four main eclipse categories: partial, annular, total,
and hybrid, with additional codes describing the eclipse's Saros-series
position or path characteristics. 
    - ( chagpt help. I dont know anything about eclipse.)


5. Columns where cleaning is decided:
   - Reason: Create 2 filters `Year` and `Eclipse Type` in the deplyoed dashboard 
   - `Calendar Date` to extract and filter by `Year`
   - Convert the 19 detailed codes into four categories: **Partial, Annular, Total, and Hybrid**

6. No cleaning is required for the other columnns
7. **GRAIN:** Each row represents one solar eclipse event. This is how I will count the total per year per eclipse.


### Check the eclipse types
- Added Link to NASA's eclipse types catalog
  - https://eclipse.gsfc.nasa.gov/SEcat5/SEcatkey.html

## Load

In [ ]:
from pathlib import Path

import pandas as pd

solar_df = pd.read_csv("../backend/data/raw/solar.csv")

solar_df.head()

,Catalog Number,Calendar Date,Eclipse Time,Delta T (s),Lunation Number,Saros Number,Eclipse Type,Gamma,Eclipse Magnitude,Latitude,Longitude,Sun Altitude,Sun Azimuth,Path Width (km),Central Duration
0,1,-1999 June 12,03:14:51,46438,-49456,5,T,-0.2701,1.0733,6.0N,33.3W,74,344,247,06m37s
1,2,-1999 December 5,23:45:23,46426,-49450,10,A,-0.2317,0.9382,32.9S,10.8E,76,21,236,06m44s
2,3,-1998 June 1,18:09:16,46415,-49444,15,T,0.4994,1.0284,46.2N,83.4E,60,151,111,02m15s
3,4,-1998 November 25,05:57:03,46403,-49438,20,A,-0.9045,0.9806,67.8S,143.8W,25,74,162,01m14s
4,5,-1997 April 22,13:19:56,46393,-49433,-13,P,-1.4670,0.1611,60.6S,106.4W,0,281,NaN,NaN


## Inspect

In [ ]:
# number of rows, columns
solar_df.shape

(11898, 15)

In [25]:
# Display all columns in list format
solar_df.columns.tolist()

['Catalog Number',
 'Calendar Date',
 'Eclipse Time',
 'Delta T (s)',
 'Lunation Number',
 'Saros Number',
 'Eclipse Type',
 'Gamma',
 'Eclipse Magnitude',
 'Latitude',
 'Longitude',
 'Sun Altitude',
 'Sun Azimuth',
 'Path Width (km)',
 'Central Duration']

In [ ]:
# Inspect int columns integer only
solar_df.describe()

,Catalog Number,Delta T (s),Lunation Number,Saros Number,Gamma,Eclipse Magnitude,Sun Altitude,Sun Azimuth
count,11898.000000,11898.000000,11898.000000,11898.000000,11898.000000,11898.000000,11898.000000,11898.000000
mean,5949.500000,12142.172802,-18546.959321,87.483190,-0.002469,0.812748,36.505295,180.264330
std,3434.801086,13583.402888,17906.572982,48.380284,0.900860,0.300398,32.417350,110.745408
min,1.000000,-6.000000,-49456.000000,-13.000000,-1.569000,0.000000,0.000000,0.000000
25%,2975.250000,970.250000,-33954.750000,47.000000,-0.786575,0.675925,0.000000,89.000000
50%,5949.500000,5636.500000,-18495.000000,87.000000,-0.003850,0.950600,38.000000,180.000000
75%,8923.750000,20943.500000,-3039.250000,128.000000,0.776900,1.018400,66.000000,272.000000
max,11898.000000,46438.000000,12378.000000,190.000000,1.570600,1.081300,90.000000,360.000000


In [ ]:
# Check duplicate rows
solar_df.duplicated().sum()

# 0 duplicated rows

np.int64(0)

In [10]:
# Check data types and missing value counts
solar_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 11898 entries, 0 to 11897
Data columns (total 15 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Catalog Number     11898 non-null  int64  
 1   Calendar Date      11898 non-null  str    
 2   Eclipse Time       11898 non-null  str    
 3   Delta T (s)        11898 non-null  int64  
 4   Lunation Number    11898 non-null  int64  
 5   Saros Number       11898 non-null  int64  
 6   Eclipse Type       11898 non-null  str    
 7   Gamma              11898 non-null  float64
 8   Eclipse Magnitude  11898 non-null  float64
 9   Latitude           11898 non-null  str    
 10  Longitude          11898 non-null  str    
 11  Sun Altitude       11898 non-null  int64  
 12  Sun Azimuth        11898 non-null  int64  
 13  Path Width (km)    7698 non-null   str    
 14  Central Duration   7698 non-null   str    
dtypes: float64(2), int64(6), str(7)
memory usage: 1.8 MB


In [ ]:
# get the exact number of missing values
solar_df.isna().sum()

Catalog Number          0
Calendar Date           0
Eclipse Time            0
Delta T (s)             0
Lunation Number         0
Saros Number            0
Eclipse Type            0
Gamma                   0
Eclipse Magnitude       0
Latitude                0
Longitude               0
Sun Altitude            0
Sun Azimuth             0
Path Width (km)      4200
Central Duration     4200
dtype: int64

In [15]:
# display only the names of columns containing at least one null value
solar_df.columns[solar_df.isna().any()].tolist()

['Path Width (km)', 'Central Duration']

### Inspect the missing values further

In [ ]:
# Check the exact rows with missing values
solar_df[solar_df["Path Width (km)"].isna()][
    ["Path Width (km)", "Central Duration"]
].head()

,Path Width (km),Central Duration
4,NaN,NaN
5,NaN,NaN
6,NaN,NaN
7,NaN,NaN
14,NaN,NaN


In [ ]:
# checking the values with the eclipse types that has missing values

solar_df[solar_df["Path Width (km)"].isna()][
    "Eclipse Type"
].value_counts()

Eclipse Type
P     3875
Pb     163
Pe     162
Name: count, dtype: int64

### Missing-value observation

`Path Width (km)` and `Central Duration` contain 4,200 missing values.  
These records are partial eclipses (`P`, `Pb`, and `Pe`), for which a central
path width and central duration are not applicable. Therefore, the missing
values are intentional and will remain as `NaN`.

## Check the eclipse types
- Added Link to NASA's solar eclipse types catalog
  - https://eclipse.gsfc.nasa.gov/SEcat5/SEcatkey.html
  
1. P, Pb, Pe       → Partial
2. A, Am, An, A+   → Annular
3. T, Tm, Tn, T+   → Total
4. H, Hm, H2, H3   → Hybrid

In [ ]:
# Dsiplay the solar eclipse types
solar_df["Eclipse Type"].value_counts()

Eclipse Type
P     3875
A     3755
T     3049
H      502
Pb     163
Pe     162
Tm      72
Am      72
An      36
A+      34
A-      34
H3      26
As      25
H2      24
Hm      17
T-      17
Tn      14
Ts      12
T+       9
Name: count, dtype: int64